# Semana 5 — Persistência SQL

Nesta versão, uso o **`fintech_integrada.db` recebido**. Antes de continuar, verifico o esquema, os relacionamentos e o conteúdo já gravado. Não vou reconstruir o banco com estudantes ou despesas criados para demonstração: a ideia é trabalhar com a mesma fonte nas próximas semanas.

O banco tem a tabela `despesas` (lançamentos de origem) e `despesas_clean` (lançamentos tratados na Semana 6). A segunda não é apenas uma cópia da primeira.


In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# Funciona ao executar o notebook pela raiz do projeto ou pela pasta da semana.
def encontrar_raiz():
    for p in (Path.cwd(), Path.cwd().parent):
        if (p / 'semana_10_app_streamlit' / 'fintech_integrada.db').is_file():
            return p.resolve()
    raise FileNotFoundError('Não encontrei semana_10_app_streamlit/fintech_integrada.db.')

RAIZ = encontrar_raiz()
BANCO = RAIZ / 'semana_10_app_streamlit' / 'fintech_integrada.db'
conexao = sqlite3.connect(f'file:{BANCO}?mode=ro', uri=True)
conexao.execute('PRAGMA foreign_keys=ON')
print('Banco utilizado:', BANCO.name)
print('Integridade:', conexao.execute('PRAGMA integrity_check').fetchone()[0])


Banco utilizado: fintech_integrada.db
Integridade: ok


## Estrutura e integridade do banco


In [2]:
tabelas=pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name",conexao)
display(tabelas)
for tabela in ('estudantes','municipios','despesas','despesas_clean'):
    print(f'\n{tabela}')
    display(pd.read_sql_query(f'PRAGMA table_info({tabela})',conexao)[['name','type','pk']])
print('Verificação de chaves estrangeiras:',conexao.execute('PRAGMA foreign_key_check').fetchall())


,name
0,despesas
1,despesas_clean
2,estudantes
3,municipios



estudantes


,name,type,pk
0,id,INTEGER,1
1,nome,TEXT,0
2,matricula,TEXT,0



municipios


,name,type,pk
0,id,INTEGER,1
1,nome,TEXT,0
2,uf_sigla,TEXT,0



despesas


,name,type,pk
0,id,INTEGER,1
1,valor,REAL,0
2,data,TEXT,0
3,estudante_id,INTEGER,0
4,municipio_id,INTEGER,0



despesas_clean


,name,type,pk
0,id,INTEGER,0
1,valor,REAL,0
2,data,TEXT,0
3,estudante_id,INTEGER,0
4,municipio_id,INTEGER,0
5,categoria,TEXT,0
6,is_outlier,INTEGER,0


Verificação de chaves estrangeiras: []


## Quantidades e comparação das duas tabelas de despesas

A tabela tratada contém registros adicionais. Por isso, não devo chamar a soma de `despesas` de "total do projeto" ou compará-la com a soma de `despesas_clean` como se fossem dois períodos.


In [3]:
resumo=pd.read_sql_query("""SELECT 'estudantes' AS tabela, COUNT(*) AS registros FROM estudantes
UNION ALL SELECT 'municipios', COUNT(*) FROM municipios
UNION ALL SELECT 'despesas', COUNT(*) FROM despesas
UNION ALL SELECT 'despesas_clean', COUNT(*) FROM despesas_clean""",conexao)
display(resumo)

fontes=pd.read_sql_query('SELECT d.id,e.nome AS estudante,d.valor,d.data,m.nome AS municipio,m.uf_sigla AS uf FROM despesas d JOIN estudantes e ON e.id=d.estudante_id JOIN municipios m ON m.id=d.municipio_id ORDER BY d.id',conexao)
tratadas=pd.read_sql_query('SELECT * FROM despesas_clean ORDER BY id',conexao)
display(fontes)
print('Soma de despesas (origem): R$',round(fontes.valor.sum(),2))
print('Soma de despesas_clean (tratada): R$',round(tratadas.valor.sum(),2))


,tabela,registros
0,estudantes,3
1,municipios,1590
2,despesas,3
3,despesas_clean,7


,id,estudante,valor,data,municipio,uf
0,1,Ana Souza,125.9,2026-09-01,Adamantina,SP
1,2,Bruno Lima,89.5,2026-09-02,Angra dos Reis,RJ
2,3,Carla Mendes,210.0,2026-09-03,Abadia dos Dourados,MG


Soma de despesas (origem): R$ 425.4
Soma de despesas_clean (tratada): R$ 5673.0


## Relacionamentos SQL: estudante, despesa e município


In [4]:
consulta = """
    SELECT d.id,d.valor,d.data,d.estudante_id,d.municipio_id,
           d.categoria,d.is_outlier,
           e.nome AS estudante,e.matricula,m.nome AS municipio,m.uf_sigla AS uf
    FROM despesas_clean AS d
    INNER JOIN estudantes AS e ON e.id=d.estudante_id
    INNER JOIN municipios AS m ON m.id=d.municipio_id
    ORDER BY d.data,d.id
"""
despesas = pd.read_sql_query(consulta,conexao,parse_dates=['data'])
despesas['is_outlier'] = despesas['is_outlier'].fillna(0).astype(bool)
display(despesas)


,id,valor,data,estudante_id,municipio_id,categoria,is_outlier,estudante,matricula,municipio,uf
0,1,125.9,2026-09-01,1,3500105,alimentação,False,Ana Souza,2026001,Adamantina,SP
1,2,89.5,2026-09-02,2,3300100,transporte,False,Bruno Lima,2026002,Angra dos Reis,RJ
2,3,210.0,2026-09-03,3,3100104,lazer,False,Carla Mendes,2026003,Abadia dos Dourados,MG
3,4,107.7,2026-09-07,1,3500105,alimentação,False,Ana Souza,2026001,Adamantina,SP
4,5,75.0,2026-09-08,1,3500105,nao_informado,False,Ana Souza,2026001,Adamantina,SP
5,6,5000.0,2026-09-09,1,3500105,transporte,True,Ana Souza,2026001,Adamantina,SP
6,7,64.9,2026-09-10,1,3500105,lazer,False,Ana Souza,2026001,Adamantina,SP


## Despesas agrupadas por estado e estudante


In [5]:
por_uf=pd.read_sql_query("""SELECT m.uf_sigla AS uf, COUNT(*) AS lancamentos, ROUND(SUM(d.valor),2) AS total
FROM despesas_clean d
JOIN municipios m ON m.id=d.municipio_id
GROUP BY m.uf_sigla ORDER BY total DESC""",conexao)
por_aluno=despesas.groupby('estudante',as_index=False).agg(lancamentos=('id','size'),total=('valor','sum'))
display(por_uf)
display(por_aluno)


,uf,lancamentos,total
0,SP,5,5373.5
1,MG,1,210.0
2,RJ,1,89.5


,estudante,lancamentos,total
0,Ana Souza,5,5373.5
1,Bruno Lima,1,89.5
2,Carla Mendes,1,210.0


## Teste de transação com rollback, sem alterar o arquivo

Para testar uma chave estrangeira inválida, crio um banco temporário **em memória**. A falha deve desfazer o `INSERT`. O arquivo enviado não sofre modificações.


In [6]:
copia=sqlite3.connect(':memory:')
conexao.backup(copia)
copia.execute('PRAGMA foreign_keys=ON')
qtd_antes=copia.execute('SELECT COUNT(*) FROM despesas').fetchone()[0]
try:
    with copia:
        # Reaproveito uma despesa existente e troco apenas a referência pela ID inválida.
        caso=fontes.iloc[0]
        copia.execute('INSERT INTO despesas(valor,data,estudante_id,municipio_id) VALUES(?,?,?,?)',
                      (float(caso.valor),str(caso.data),999999999,3500105))
except sqlite3.IntegrityError:
    print('Chave estrangeira inválida rejeitada. Rollback realizado.')
qtd_depois=copia.execute('SELECT COUNT(*) FROM despesas').fetchone()[0]
assert qtd_antes==qtd_depois
print('Registros antes/depois:',qtd_antes,qtd_depois)
copia.close()
conexao.close()


Chave estrangeira inválida rejeitada. Rollback realizado.
Registros antes/depois: 3 3


## Conclusão

O banco passou nas verificações de integridade e permite relacionar despesas, estudantes e municípios com `JOIN`. A análise estatística das próximas semanas usará os **sete registros de `despesas_clean`**, preservando as três linhas de `despesas` como fonte histórica.
